# 01 — Demand Data Exploration

In [5]:
!git clone https://github.com/hemanthkarthikd/demand-forecasting-inventory-optimization.git
%cd demand-forecasting-inventory-optimization/notebooks

Cloning into 'demand-forecasting-inventory-optimization'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 25 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 44.71 KiB | 1.15 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/demand-forecasting-inventory-optimization/notebooks/demand-forecasting-inventory-optimization/notebooks


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

demand = pd.read_csv("../data/raw/weekly_demand.csv", parse_dates=["week_start"])
sku = pd.read_csv("../data/raw/sku_master.csv")

print("Demand shape:", demand.shape)
print("SKU master shape:", sku.shape)
display(demand.head())
display(sku.head())

Demand shape: (3456, 6)
SKU master shape: (24, 10)


,week_start,sku_id,sku_name,category,demand_units,promo_flag
0,2024-01-01,SKU-001,Electronics Item 1,Electronics,66,0
1,2024-01-08,SKU-001,Electronics Item 1,Electronics,80,0
2,2024-01-15,SKU-001,Electronics Item 1,Electronics,67,0
3,2024-01-22,SKU-001,Electronics Item 1,Electronics,63,0
4,2024-01-29,SKU-001,Electronics Item 1,Electronics,63,0


,sku_id,sku_name,category,unit_cost,selling_price,lead_time_weeks,ordering_cost,annual_holding_rate,target_service_level,demand_profile
0,SKU-001,Electronics Item 1,Electronics,27.97,49.97,5,148.95,0.274,0.95,Regular
1,SKU-002,Electronics Item 2,Electronics,74.98,125.56,3,98.58,0.191,0.97,Regular
2,SKU-003,Electronics Item 3,Electronics,169.66,237.32,5,97.34,0.218,0.95,Regular
3,SKU-004,Electronics Item 4,Electronics,200.31,255.31,4,87.97,0.239,0.95,Regular
4,SKU-005,Electronics Item 5,Electronics,72.04,105.67,3,104.16,0.180,0.95,Regular


In [7]:
print("Missing values:")
display(demand.isna().sum())
print("Duplicate rows:", demand.duplicated().sum())
print("Negative demand rows:", (demand["demand_units"] < 0).sum())
print("Unique SKUs:", demand["sku_id"].nunique())
print("Date range:", demand["week_start"].min(), "to", demand["week_start"].max())

Missing values:


,0
week_start,0
sku_id,0
sku_name,0
category,0
demand_units,0
promo_flag,0


Duplicate rows: 0
Negative demand rows: 0
Unique SKUs: 24
Date range: 2024-01-01 00:00:00 to 2026-09-28 00:00:00


In [8]:
profile = (
    demand.groupby(["sku_id","sku_name","category"])
    .agg(
        avg_weekly_demand=("demand_units","mean"),
        demand_std=("demand_units","std"),
        total_demand=("demand_units","sum"),
        zero_demand_weeks=("demand_units", lambda s: (s==0).sum())
    ).reset_index()
)
profile["cv"] = profile["demand_std"] / profile["avg_weekly_demand"].replace(0, np.nan)
profile = profile.merge(sku[["sku_id","unit_cost"]], on="sku_id")
profile["annual_dollar_usage"] = profile["avg_weekly_demand"]*52*profile["unit_cost"]

profile = profile.sort_values("annual_dollar_usage", ascending=False).reset_index(drop=True)
profile["spend_share"] = profile["annual_dollar_usage"]/profile["annual_dollar_usage"].sum()
profile["cumulative_spend_share"] = profile["spend_share"].cumsum()
profile["abc_class"] = np.select(
    [profile["cumulative_spend_share"] <= 0.80,
     profile["cumulative_spend_share"] <= 0.95],
    ["A","B"], default="C"
)
profile["xyz_class"] = np.select(
    [profile["cv"] <= 0.35, profile["cv"] <= 0.70],
    ["X","Y"], default="Z"
)

profile.to_csv("../data/processed/sku_demand_profile.csv", index=False)
display(profile)

,sku_id,sku_name,category,avg_weekly_demand,demand_std,total_demand,zero_demand_weeks,cv,unit_cost,annual_dollar_usage,spend_share,cumulative_spend_share,abc_class,xyz_class
0,SKU-017,Packaging Item 5,Packaging,254.708333,46.370487,36678,0,0.182053,212.37,2.812805e+06,0.128837,0.128837,A,X
1,SKU-003,Electronics Item 3,Electronics,317.944444,64.412979,45784,0,0.202592,169.66,2.805008e+06,0.128480,0.257317,A,X
2,SKU-004,Electronics Item 4,Electronics,243.756944,45.055560,35101,0,0.184838,200.31,2.539002e+06,0.116296,0.373613,A,X
3,SKU-006,Electronics Item 6,Electronics,181.145833,152.736895,26085,52,0.843171,210.04,1.978489e+06,0.090622,0.464235,A,Z
4,SKU-010,Fasteners Item 4,Fasteners,211.048611,42.227047,30391,0,0.200082,147.60,1.619840e+06,0.074195,0.538430,A,X
5,SKU-019,Plastics Item 1,Plastics,201.215278,42.608834,28975,0,0.211757,123.19,1.288961e+06,0.059039,0.597469,A,X
6,SKU-002,Electronics Item 2,Electronics,290.229167,80.419157,41793,0,0.277088,74.98,1.131592e+06,0.051831,0.649300,A,X
7,SKU-018,Packaging Item 6,Packaging,163.715278,138.271379,23575,54,0.844584,125.24,1.066192e+06,0.048836,0.698136,A,Z
8,SKU-005,Electronics Item 5,Electronics,199.013889,49.071945,28658,0,0.246575,72.04,7.455219e+05,0.034148,0.732284,A,X
9,SKU-022,Plastics Item 4,Plastics,202.416667,44.664797,29148,0,0.220658,70.12,7.380597e+05,0.033806,0.766089,A,X
